# Delete Files Above a Comparison Threshold

Set `MIN_COMPARISONS`, `DRY_RUN`, and `LIMIT`, then run the first cell to select files. Run the second cell to delete the selected JSON files. The limit applies from the highest comparison count down.

In [ ]:
from comfyui_image_scorer.adapters.cli.deps import build_cli_deps
from comfyui_image_scorer.infrastructure.persistence.path_handler import (
    compute_path_from_filename,
)

MIN_COMPARISONS = 35
DRY_RUN = False
LIMIT = 5
print("creating graph...")
graph = build_cli_deps().graph
print("rebuilding graph from database...")
graph.rebuild_from_database()
print("getting all nodes...")
all_nodes = graph.get_all_nodes()
print(
    f"graph rebuilt, {len(all_nodes)} nodes found, filtering nodes with more than {MIN_COMPARISONS} comparisons..."
)
selected_nodes = sorted(
    (node for node in all_nodes if node.link_count > MIN_COMPARISONS),
    key=lambda node: node.link_count,
    reverse=True,
)
if LIMIT is not None:
    selected_nodes = selected_nodes[:LIMIT]

print(f"Total files to delete: {len(selected_nodes)}, computing paths...")

files_to_delete = [
    (
        node.link_count,
        compute_path_from_filename(node.id, node.score).with_suffix(".json"),
    )
    for node in selected_nodes
]

print(
    f"Mode: {'DRY RUN' if DRY_RUN else 'DELETE'}; threshold: > {MIN_COMPARISONS}; limit: {LIMIT}"
)
print(f"Total files to delete: {len(files_to_delete)}")
# for comparison_count, json_path in files_to_delete:
#     print(f"{comparison_count} comparisons: {json_path}")


In [ ]:
for comparison_count, json_path in files_to_delete:
    if DRY_RUN:
        print(f"Would delete ({comparison_count} comparisons): {json_path}")
    else:
        json_path.unlink(missing_ok=True)
        print(f"Deleted ({comparison_count} comparisons): {json_path}")
